Inputs: Training Set (Data, Labels)
Outputs: Metrics: Loss and Accuracy

In [ ]:
# imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

from datasets.download_datasets import *
from utils.training_utils import *

In [ ]:
# args/configs
d_args = {
    "filts": [[1, 32], [32, 32], [32, 64], [64, 64], [64, 128]], # Example AASIST filter bank
    "gat_dims": [64, 32],
    "pool_ratios": [0.5, 0.7, 0.5],
    "temperatures": [2.0, 2.0, 1.0],
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 8  # Keep small for W2V2
lr = 0.0001
epochs = 10

In [ ]:
# wandb and huggingface setup
import wandb
from huggingface_hub import HfApi, login

# Initialize W&B
wandb.init(
    project="audio-deepfake-detection",
    config={
        "learning_rate": lr,
        "architecture": "W2V2_AASIST",
        "dataset": "SpeechFake",
        "epochs": epochs,
    }
)

# Login to HF (Run this once or use a token)
# login() 
api = HfApi()
repo_id = "your-username/w2v2-aasist-checkpoints"

In [ ]:
# initialize DataLoader
train_data, val_data = get_speechfake(splits=['train', 'validation'])
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_data, batch_size=batch_size, collate_fn=collate_fn)

In [ ]:
# import model and initialize optimizer
from baseline.w2v2_aasist import W2V2_AASIST

model = W2V2_AASIST(d_args).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

In [ ]:
# train
for epoch in range(epochs):
    loss = train_epoch(model, train_loader, optimizer, criterion, device)
    loss, acc = validate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: Loss {loss:.4f}, Val Acc {acc:.4f}")